# RAGAS, @retry, and Pydantic — Concept Deep Dive

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
for candidate in [project_root, *project_root.parents]:
    if (candidate / "src").exists():
        project_root = candidate
        break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root added to sys.path: {project_root}")

## Part 1: What is RAGAS?

### The metrics we use in this project (`src/backend/evaluation/ragas_eval.py`)

### Reference-free vs reference-based

In [ ]:
# TODO: import the pieces we need for the RAGAS demos below.
#
# 1. from ragas.llms import LangchainLLMWrapper
# 2. from ragas import SingleTurnSample
# 3. from ragas.embeddings import LangchainEmbeddingsWrapper
# 4. from ragas.metrics import Faithfulness, LLMContextPrecisionWithoutReference,
#    ResponseRelevancy, LLMContextRecall, FactualCorrectness
# 5. import asyncio
#
# 6. from src.backend.rag.llm import get_llm
# 7. from src.backend.rag.embeddings import get_embeddings


print("Imports ready -- same imports as the top of ragas_eval.py")

### `SingleTurnSample` — RAGAS's data contract

In [ ]:
# TODO: build the test data for the Faithfulness demo.
#
# 1. context            -> a list with ONE string that states a true fact about RAG
# 2. question           -> "What is the main benefit of RAG?"
# 3. grounded_answer     -> an answer that correctly paraphrases `context`
# 4. hallucinated_answer -> an answer that is confidently WRONG / made up
#
# 5. Wrap each answer in a SingleTurnSample (user_input=question, response=..., retrieved_contexts=context):
#    sample_grounded = SingleTurnSample(...)
#    sample_hallucinated = SingleTurnSample(...)


print("Two samples ready: one grounded answer, one hallucinated answer, same question and context")

### Wrapping your LLM for RAGAS

In [ ]:
# TODO:
# 1. llm = get_llm()
# 2. evaluator_llm = LangchainLLMWrapper(llm)


print("LLM wrapped for RAGAS")

### Demo 1 — Faithfulness catches hallucination

In [ ]:
# TODO: score both samples with the Faithfulness metric.
#
# 1. Write an async function score_faithfulness(sample) that:
#       - creates metric = Faithfulness(llm=evaluator_llm)
#       - returns await metric.single_turn_ascore(sample)
# 2. grounded_score = asyncio.run(score_faithfulness(sample_grounded))
# 3. hallucinated_score = asyncio.run(score_faithfulness(sample_hallucinated))
# 4. Print both scores, then print (1 - hallucinated_score) as the hallucination score



### Demo 2 — Context Precision (reference-free)

In [ ]:
# TODO: score Context Precision on sample_grounded.
#
# 1. Write an async function score_context_precision(sample) using
#    metric = LLMContextPrecisionWithoutReference(llm=evaluator_llm)
# 2. Run it with asyncio.run(...) and print the result



### Demo 3 — Response Relevancy (needs embeddings)

In [ ]:
# TODO: score Response Relevancy -- this one needs embeddings too.
#
# 1. embeddings = get_embeddings("OpenAI (text-embedding-3-small)")
# 2. evaluator_embeddings = LangchainEmbeddingsWrapper(embeddings)
# 3. Write an async function score_relevancy(sample) using
#    metric = ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)
# 4. Run it on sample_grounded with asyncio.run(...) and print the result



### Demo 4 — Reference-based metrics: Context Recall + Factual Correctness

In [ ]:
# TODO: reference-based metrics need a gold reference answer.
#
# 1. reference_answer -> write a gold-standard answer to `question`
# 2. sample_with_reference = SingleTurnSample(user_input=question, response=grounded_answer,
#       retrieved_contexts=context, reference=reference_answer)
# 3. Write async functions score_recall(sample) using LLMContextRecall(llm=evaluator_llm)
#    and score_factual_correctness(sample) using FactualCorrectness(llm=evaluator_llm)
# 4. Run both on sample_with_reference with asyncio.run(...) and print the results



### Demo 5 — LLM-as-a-Judge (the non-RAGAS evaluator in our project)

In [ ]:
# TODO: build your own judge -- a prompt + structured JSON output.
#
# 1. from langchain_core.prompts import PromptTemplate
#    from langchain_core.output_parsers import JsonOutputParser
# 2. Build judge_prompt with input_variables=["query", "context_str", "response"].
#    The template should ask the LLM to judge RELEVANCY and respond ONLY in JSON with keys:
#    relevancy_percentage, explanation, hallucination_detected
# 3. judge_chain = judge_prompt | llm | JsonOutputParser()
# 4. Invoke judge_chain on {"query": question, "context_str": "\n".join(context), "response": hallucinated_answer}
# 5. Print the result



### 🔎 Compare with the real file: `src/backend/evaluation/ragas_eval.py`

## Part 2: What does `@retry` mean?

### Demo 1 — without retry: one blip kills the whole request

In [ ]:
# TODO: simulate a flaky API that fails twice, then succeeds on the 3rd try.
#
# 1. attempt_counter = {"count": 0}   (a dict, not a plain int, so it survives between calls)
# 2. def call_flaky_api():
#       - increment attempt_counter["count"]
#       - print the attempt number
#       - if count < 3: raise ConnectionError("Simulated network blip")
#       - else: return "success"
# 3. Call call_flaky_api() inside a try/except ConnectionError and print what happens



### Demo 2 — same function, now with `tenacity`'s `@retry`

In [ ]:
# TODO: wrap the same kind of function with tenacity's @retry.
#
# 1. from tenacity import retry, stop_after_attempt, wait_exponential
# 2. Reset attempt_counter["count"] = 0
# 3. Write call_flaky_api_with_retry(), decorated with:
#       @retry(stop=stop_after_attempt(3),
#              wait=wait_exponential(multiplier=1, min=2, max=10),
#              reraise=True)
#    (same fail-twice-then-succeed body as the previous demo)
# 4. Call it and print the result



### Breaking down the parameters

### Demo 3 — what happens when the API is genuinely down (retries exhausted)

In [ ]:
# TODO: what happens when retries run out because the API is genuinely down?
#
# 1. Reset attempt_counter["count"] = 0
# 2. Write always_fails(), decorated with:
#       @retry(stop=stop_after_attempt(3),
#              wait=wait_exponential(multiplier=1, min=1, max=3),
#              reraise=True)
#    Inside, always raise ConnectionError("This API is genuinely down")
# 3. Call it inside try/except ConnectionError and print the message after attempts are exhausted



### 🔎 Compare with the real files

## Part 3: What is a Pydantic schema?

### Demo 1 — a plain class hides a bug

In [ ]:
# TODO: build a plain (non-validating) Python class.
#
# 1. class PlainQueryRequest:
#       __init__(self, question, embedding_model, retriever_type) stores each as self.attribute
# 2. bad_request = PlainQueryRequest(question=123, embedding_model="OpenAI", retriever_type="FAISS Retriever")
#    (question=123 is an int, on purpose)
# 3. Print bad_request.question and type(bad_request.question) -- notice nothing complains



### Demo 2 — the same shape, with Pydantic's `BaseModel`

In [ ]:
# TODO: rebuild the same shape using Pydantic's BaseModel.
#
# 1. from pydantic import BaseModel
# 2. class QueryRequest(BaseModel):
#       question: str
#       embedding_model: str
#       retriever_type: str
#       top_k: int = 5
# 3. good_request = QueryRequest(question="What is RAG?", embedding_model="OpenAI", retriever_type="FAISS Retriever")
#    Print it.
# 4. Inside try/except, create bad_request = QueryRequest(..., top_k="five")  (not coercible to int)
#    Print the validation error



### FastAPI uses this exact pattern for the request/response contract

In [ ]:
# TODO: build a response schema the same way.
#
# 1. class QueryResponse(BaseModel):
#       answer: str
#       answer_texts: list
#       evaluation_metric: dict
# 2. response = QueryResponse(answer=..., answer_texts=[...], evaluation_metric={...})
# 3. Print response.model_dump_json(indent=2)



### `BaseSettings` — Pydantic for configuration

In [ ]:
# TODO: Pydantic for configuration -- BaseSettings reads from environment variables.
#
# 1. import os ; from pydantic_settings import BaseSettings ; from pydantic import Field
# 2. os.environ["DEMO_API_KEY"] = "sk-demo-12345"   (simulating a .env file)
# 3. class DemoSettings(BaseSettings):
#       DEMO_API_KEY: str = Field("", env="DEMO_API_KEY")
#       DEMO_TIMEOUT: int = Field(30, env="DEMO_TIMEOUT")
# 4. demo_settings = DemoSettings() ; print it
#    (DEMO_API_KEY should come from the env var, DEMO_TIMEOUT should fall back to its default of 30)



### 🔎 Compare with the real files

## Now walk through the real production files